In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [10]:
#Ví dụ linh hồn của Transformer: Multi-Head Attention với cơ chế Query, Key, Value
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.d_k = d_model // n_heads # 64 chiều mỗi đầu
        self.n_heads = n_heads
        
        #Các ma trận trọng số WQ, WK, WV
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        #Tạo Q, K, V và chia thành n_heads
        Q = self.w_q(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        #Tính Scores bằng tích vô hướng và Scaling
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        #Softmax để lấy trọng số chú ý
        attn = F.softmax(scores, dim=-1)
        
        #Nhân với Value và gộp các đầu lại
        context = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.w_o(context)

In [11]:
#Embedding Size_01: (batch_size=1, seq_len=10, d_model=512)
sample_input = torch.randn(1, 10, 512) 

#Khởi tạo module Multi-Head Attention với 8 đầu (Heads)
mha = MultiHeadAttention(d_model=512, n_heads=8)

#Chạy thực tế (với Q, K, V đều là input ban đầu trong Self-Attention)
context_vectors = mha(sample_input, sample_input, sample_input)

print(f"INPUT (1): {sample_input.shape}")
print(f"OUTPUT (Context Vectors): {context_vectors.shape}")

INPUT (1): torch.Size([1, 10, 512])
OUTPUT (Context Vectors): torch.Size([1, 10, 512])
